# Sentiment Analysis: Building and Benchmarking NLP Classifiers from Scratch

This project builds a complete sentiment analysis pipeline on the Stanford Sentiment Treebank (SST) dataset — classifying movie reviews as positive (+1) or negative (−1). Rather than relying solely on high-level library abstractions, the goal is to build key components from scratch, understand the tradeoffs between different linear models, and rigorously analyze which approaches scale to real-world data sizes.

**The pipeline covers:**
- Custom NLP featurization via Bag-of-Words
- Benchmarking 4 sklearn classifiers across accuracy, speed, and generalization
- Implementing Gradient Descent and SGD from scratch in NumPy
- Scalability analysis on the full 40,000+ example dataset

## 1. NLP Pipeline & Featurization

### 1.1 Loading the Data

The dataset is the Stanford Sentiment Treebank — a collection of movie review sentences with human-annotated sentiment labels. Labels are mapped from {0, 1} to {−1, +1} to align with the linear model convention used throughout this project.

The data is split into three sets:
- **Training set (60%)** — used to fit all models
- **Validation set (20%)** — used to tune hyperparameters such as learning rate and regularization strength, without touching the test set
- **Test set (20%)** — held out until final evaluation only

This three-way split is critical. Using the test set for any decision during development would give an overly optimistic estimate of real-world performance.

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split

df = pd.read_csv("data.tsv", sep="\t")
df["label"] = df["label"].map({0: -1, 1: 1})  # map 0 to -1, 1 remains 1

df_train, df_tmp = train_test_split(
    df,
    test_size=0.4,
    random_state=42,
    shuffle=True
)

df_val, df_test = train_test_split(
    df_tmp,
    test_size=0.5
)

print(df_train.head())
print()
print(df_train.columns)

### 1.2 Tokenization & Vocabulary Construction

Text cannot be fed directly into a linear model — it must first be converted into a numerical vector. The process starts with **tokenization**: splitting each sentence into individual words (lowercase, retaining apostrophes for contractions like *don't*).

A vocabulary is then built from all unique words in the **training set only**. Using the training set to build the vocabulary is important — including test set words would leak information about the test distribution.

In [ ]:
import re
from collections import Counter

def tokenize(text):
    text = str(text).lower()
    # words contain alphabetic characters and apostrophes (e.g. don't)
    return re.findall(r"[a-zA-Z]+(?:'[a-zA-Z]+)?", text)

vocab_counter = Counter()

for sent in df_train["sentence"]:
    tokens = tokenize(sent)
    vocab_counter.update(tokens)

vocab = sorted(vocab_counter.keys())
vocab_size = len(vocab)

print("Vocabulary size:", vocab_size)

# map each word to a unique index
word2idx = {word: i for i, word in enumerate(vocab)}

### 1.3 Bag-of-Words Encoding

Each sentence is encoded as a **Bag-of-Words (BoW)** vector — a 1D array of length equal to the vocabulary size (13,297), where each entry counts how many times the corresponding word appears in the sentence.

This representation discards word order but captures word frequency, which is sufficient for sentiment classification. The result is a high-dimensional sparse feature matrix: most words don't appear in any given sentence, so most entries are zero.

**Feature vector dimension: 13,297** — one dimension per unique word in the training vocabulary.

The **majority class baseline** for this dataset is **55.5%** — the fraction of training labels that are +1. Any model that cannot exceed this is not learning anything meaningful.

In [ ]:
import numpy as np

def encode_sentence_to_counts(sentence, word2idx, vocab_size):
    tokens = tokenize(sentence)
    vec = np.zeros(vocab_size, dtype=np.int32)
    for tok in tokens:
        idx = word2idx.get(tok)
        if idx is not None:
            vec[idx] += 1
    return vec

def get_X_y(df):
    X = np.vstack([
        encode_sentence_to_counts(sent, word2idx, vocab_size)
        for sent in df["sentence"]])
    y = df["label"].astype(float).to_numpy()
    return X, y

X_train, y_train = get_X_y(df_train)
X_val, y_val = get_X_y(df_val)
X_test, y_test = get_X_y(df_test)

print("Array shapes:")
print(f"X_train : {X_train.shape}, y_train : {y_train.shape}")
print(f"X_val   : {X_val.shape},   y_val   : {y_val.shape}")
print(f"X_test  : {X_test.shape},  y_test  : {y_test.shape}")
print()
print(f"Majority class baseline: {(y_train == 1).mean():.4f}")

## 2. Model Benchmarking

Four sklearn linear models are trained and compared on a 5,000-example random subset of the training data. The subset is used here because some models (particularly linear regression) do not scale to the full dataset within practical time limits — this is explored in detail in Section 4.

All models output `sign(w·x)` at test time. The key differences are in how they are trained — specifically, what loss function they minimize.

In [ ]:
def random_subset(X, y, n_samples, random_state=42):
    rng = np.random.default_rng(random_state)
    assert n_samples <= X.shape[0]
    indices = rng.choice(X.shape[0], size=n_samples, replace=False)
    return X[indices], y[indices]

X_train_sub, y_train_sub = random_subset(X_train, y_train, n_samples=5000)

### 2.1 Linear Regression

Linear regression minimizes the **Mean Squared Error (MSE)** loss — the average squared difference between predictions and true labels. It was not designed for classification, but it provides a useful baseline and introduces key evaluation metrics.

Three metrics are reported:
- **MSE** — average squared prediction error
- **R²** — fraction of label variance explained by the model. R² = 1 is perfect, R² = 0 means the model is no better than predicting the mean, and R² < 0 means the model is actively worse than predicting the mean
- **Accuracy** — fraction of correctly classified examples using `sign(prediction)`

In [ ]:
from sklearn.metrics import mean_squared_error, r2_score, accuracy_score
from sklearn.linear_model import LinearRegression

lin_reg = LinearRegression().fit(X_train_sub, y_train_sub)

def linreg_metrics(X, y, model):
    y_pred = model.predict(X)
    print(f"mse = {mean_squared_error(y, y_pred):.2f},  "
          f"r2 = {r2_score(y, y_pred):.2f},  "
          f"accuracy = {accuracy_score(y, np.sign(y_pred)):.2f}")

print("Train:", end='\t')
linreg_metrics(X_train_sub, y_train_sub, lin_reg)
print("Test: ", end='\t')
linreg_metrics(X_test, y_test, lin_reg)

**Analysis:** Linear regression achieves 100% training accuracy but only 68% on test — a clear sign of **overfitting**. With 13,297 parameters but only 5,000 training examples, the model memorizes noise rather than learning generalizable patterns. The negative R² on test (−2.99) confirms the model's raw predictions are poorly calibrated on unseen data, even though `sign()` still classifies 68% correctly.

### 2.2 Ridge Regression

Ridge regression adds an **ℓ₂ regularization term** to the MSE loss, penalizing large weights:

$$\text{Loss} = \frac{1}{n}\sum_i (y_i - \mathbf{w} \cdot \mathbf{x}_i)^2 + \lambda\|\mathbf{w}\|^2$$

The hyperparameter λ (called `alpha` in sklearn) controls the strength of regularization. Three values are compared below.

In [ ]:
from sklearn.linear_model import Ridge

for alpha in [0.1, 1.0, 10.0]:
    ridge = Ridge(alpha=alpha)
    ridge.fit(X_train_sub, y_train_sub)
    print(f"alpha = {alpha}")
    print("  Train:", end='\t')
    linreg_metrics(X_train_sub, y_train_sub, ridge)
    print("  Test: ", end='\t')
    linreg_metrics(X_test, y_test, ridge)
    print()

**Analysis:** Both `alpha = 1` and `alpha = 10` yield better test results than plain linear regression (78% vs 68%), confirming that regularization effectively reduces overfitting.

`alpha = 10` is the best performer: it sacrifices some training accuracy (93% vs 99%) but achieves a marginally better test R² (0.33 vs 0.29) with the same test accuracy. The weights are shrunk more aggressively toward zero, preventing any single word from dominating predictions on unseen data.

`alpha = 0.1` does not improve over plain linear regression — regularization is too weak to prevent overfitting, and the negative test R² (−0.07) confirms the model is barely better than predicting the mean on unseen data. This illustrates the **bias-variance tradeoff** directly: lower alpha → lower bias but higher variance → poor generalization.

### 2.3 Logistic Regression

Logistic regression is the first model designed specifically for binary classification. Instead of predicting a raw number, it outputs the **probability** that a review is positive, using the sigmoid function:

$$\Pr[+1 \mid \mathbf{x}] = \sigma(\mathbf{w} \cdot \mathbf{x}) = \frac{1}{1 + e^{-\mathbf{w} \cdot \mathbf{x}}}$$

The model is trained by maximizing the likelihood of the observed labels — equivalently, minimizing the **logistic loss** (cross-entropy):

$$\text{Loss} = \sum_i \log\left(1 + e^{-y_i \mathbf{w} \cdot \mathbf{x}_i}\right)$$

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import log_loss

def logreg_metrics(X, y, model):
    y_proba = model.predict_proba(X)
    y_pred = model.predict(X)
    print(f"log loss = {log_loss(y, y_proba):.2f},  "
          f"accuracy = {accuracy_score(y, y_pred):.2f}")

log_reg = LogisticRegression().fit(X_train_sub, y_train_sub)

print("Train:", end='\t')
logreg_metrics(X_train_sub, y_train_sub, log_reg)
print("Test: ", end='\t')
logreg_metrics(X_test, y_test, log_reg)

### 2.4 Support Vector Classifier

The Support Vector Machine finds the decision boundary that **maximizes the margin** between the two classes. It uses the **hinge loss**:

$$\text{Loss} = \sum_i \max(0, 1 - y_i \mathbf{w} \cdot \mathbf{x}_i)$$

Unlike logistic regression, which keeps adjusting weights for all examples, SVM incurs zero loss once a point is correctly classified with sufficient margin. The model focuses its learning on the hard, ambiguous examples near the decision boundary — the **support vectors**.

In [ ]:
from sklearn.svm import LinearSVC
from sklearn.metrics import hinge_loss

def svc_metrics(X, y, model):
    y_pred = model.predict(X)
    print(f"hinge loss = {hinge_loss(y, y_pred):.4f},  "
          f"accuracy = {accuracy_score(y, y_pred):.2f}")

svc = LinearSVC().fit(X_train_sub, y_train_sub)

print("Train:", end='\t')
svc_metrics(X_train_sub, y_train_sub, svc)
print("Test: ", end='\t')
svc_metrics(X_test, y_test, svc)

### 2.5 Model Comparison Summary

| Model | Train Accuracy | Test Accuracy | Training Time |
|---|---|---|---|
| Linear Regression | 100% | 68% | ~113s |
| Ridge Regression (α=10) | 93% | 78% | ~16s |
| Logistic Regression | 97% | 79% | ~10s |
| LinearSVC | 99% | 79% | ~2s |

**Speed:** Linear regression is the slowest by a wide margin due to its O(d³) matrix inversion — constructing and inverting a 13,297 × 13,297 matrix on 5,000 examples is expensive. Ridge is faster because ℓ₂ regularization makes the matrix better conditioned, reducing numerical instability. Logistic regression uses iterative gradient descent internally, which is cheaper than exact matrix solvers. SVM is the fastest because it only updates weights based on support vectors — once a point clears the margin, it contributes zero gradient and zero computation.

**Accuracy:** All classification-specific models (logistic regression and SVM) outperform plain linear regression on test data (79% vs 68%), demonstrating the value of using loss functions designed for binary classification. Despite their different training philosophies, both land at the same test accuracy — consistent with the fact that all models find a linear decision boundary `sign(w·x) = 0`; the difference is only in how they train.

## 3. From-Scratch Implementations

### 3.1 Gradient Descent

Gradient descent is the general optimization algorithm underlying most ML models. Instead of solving for optimal weights analytically (which is O(d³) and memory-intensive), it iteratively moves the weights in the direction that reduces the loss:

$$\mathbf{w} \leftarrow \mathbf{w} - \eta \nabla L(\mathbf{w})$$

For the MSE loss, the gradient with respect to **w** and bias **b** are:

$$\nabla_\mathbf{w} L = \frac{1}{n} X^T (\mathbf{\hat{y}} - \mathbf{y}), \quad \nabla_b L = \frac{1}{n} \sum_i (\hat{y}_i - y_i)$$

The **learning rate** η controls step size — too small and convergence is slow, too large and the loss diverges.

In [ ]:
class LinearRegressionGD:

    def __init__(self, lr=1.0, n_iters=500):
        self.lr = lr
        self.n_iters = n_iters
        self.w = None
        self.b = None
        self.losses = []

    def fit(self, X, y, verbose=False):
        y = np.asarray(y, dtype=float)
        n_samples, n_features = X.shape

        self.w = np.zeros(n_features, dtype=float)
        self.b = 0.0
        self.losses = []

        for it in range(self.n_iters):
            y_pred = X.dot(self.w) + self.b
            error = y_pred - y

            mse = np.mean(error ** 2)
            self.losses.append(mse)

            grad_w = X.T.dot(error) / len(error)
            grad_b = np.mean(error)

            self.w -= self.lr * grad_w
            self.b -= self.lr * grad_b

            if verbose and (it % 100 == 0 or it == self.n_iters - 1):
                print(f"iter {it:4d}: MSE = {mse:.4f}")

        return self

    def predict(self, X):
        assert self.w is not None and self.b is not None
        return X.dot(self.w) + self.b

#### Learning Rate Experiment

The learning rate is the most critical hyperparameter in gradient descent. Values of the form 10ᵏ for k ∈ {−2, −1, 0, 1, 2} are tested below. The validation set is used to select the best value — the test set is not touched until final evaluation.

In [ ]:
for lr in [0.01, 0.1, 1.0, 10.0, 100.0]:
    print(f"\n--- lr = {lr} ---")
    model = LinearRegressionGD(lr=lr, n_iters=500)
    model.fit(X_train_sub, y_train_sub, verbose=True)
    print("Val:  ", end='\t')
    try:
        linreg_metrics(X_val, y_val, model)
    except Exception as e:
        print(f"Failed: {e}")

In [ ]:
# Best model: lr = 1.0 — report final test performance
best_gd = LinearRegressionGD(lr=1.0, n_iters=500)
best_gd.fit(X_train_sub, y_train_sub, verbose=True)
print("Train:", end='\t')
linreg_metrics(X_train_sub, y_train_sub, best_gd)
print("Val:  ", end='\t')
linreg_metrics(X_val, y_val, best_gd)
print("Test: ", end='\t')
linreg_metrics(X_test, y_test, best_gd)

**Analysis:** Learning rates of 0.01 and 0.1 are too low — the model converges too slowly, still clearly decreasing at iteration 500 and achieving only 70-75% validation accuracy. Learning rates of 10 and 100 cause the loss to diverge completely, with weights exploding to infinity as each step overshoots the minimum.

`lr = 1.0` is the optimal value: the MSE drops from 1.0 to 0.36 in 500 iterations, achieving **78% test accuracy** — matching Ridge regression. The model is still slowly converging at iteration 500, suggesting more iterations would yield marginal further improvement.

### 3.2 Stochastic Gradient Descent (SGD)

Full batch gradient descent computes the gradient over the entire training set each iteration — expensive and slow. SGD instead computes an approximate gradient on a small random **mini-batch** of examples and updates immediately:

- One full pass through the data is called an **epoch**
- With batch size 10 and 5,000 examples, SGD makes **500 weight updates per epoch** vs **1 update per iteration** in full batch GD
- The noisier gradient is a worthwhile tradeoff — far more progress per second of computation

The gradient formulas are identical to full batch GD, just applied to the current batch `Xb, yb` instead of the full dataset.

In [ ]:
class LinearRegressionSGD:

    def __init__(self, lr=1e-3, n_epochs=10, batch_size=1, shuffle=True):
        self.lr = lr
        self.n_epochs = n_epochs
        self.batch_size = batch_size
        self.shuffle = shuffle
        self.w = None
        self.b = None
        self.losses = []

    def fit(self, X, y, verbose=False):
        y = np.asarray(y, dtype=float)
        n_samples, n_features = X.shape

        self.w = np.zeros(n_features, dtype=float)
        self.b = 0.0
        self.losses = []

        indices = np.arange(n_samples)

        for epoch in range(self.n_epochs):
            if self.shuffle:
                np.random.shuffle(indices)

            epoch_loss = 0.0

            for start in range(0, n_samples, self.batch_size):
                batch_idx = indices[start:start + self.batch_size]
                Xb = X[batch_idx]
                yb = y[batch_idx]

                error = Xb.dot(self.w) + self.b - yb

                grad_w = Xb.T.dot(error) / len(error)
                grad_b = np.mean(error)

                self.w -= self.lr * grad_w
                self.b -= self.lr * grad_b

                epoch_loss += np.mean(error ** 2) * len(batch_idx)

            epoch_loss /= n_samples
            self.losses.append(epoch_loss)

            if verbose:
                print(f"epoch {epoch + 1:3d}: MSE = {epoch_loss:.4f}")

        return self

    def predict(self, X):
        assert self.w is not None and self.b is not None
        return X.dot(self.w) + self.b

#### Learning Rate Experiment

In [ ]:
for lr in [0.01, 0.1, 1.0, 10.0, 100.0]:
    print(f"\n--- lr = {lr} ---")
    model = LinearRegressionSGD(lr=lr, n_epochs=20, batch_size=10)
    model.fit(X_train_sub, y_train_sub, verbose=True)
    print("Val:  ", end='\t')
    try:
        linreg_metrics(X_val, y_val, model)
    except Exception as e:
        print(f"Failed — likely diverged: {e}")

In [ ]:
# Best model: lr = 0.1 — report final test performance
best_sgd = LinearRegressionSGD(lr=0.1, n_epochs=20, batch_size=10)
best_sgd.fit(X_train_sub, y_train_sub, verbose=True)
print("Train:", end='\t')
linreg_metrics(X_train_sub, y_train_sub, best_sgd)
print("Test: ", end='\t')
linreg_metrics(X_test, y_test, best_sgd)

**Analysis:** `lr = 0.01` converges too slowly — MSE only drops from 1.0 to 0.63 across 20 epochs, still clearly decreasing at the end. `lr = 1.0`, `10`, and `100` all diverge, with the loss exploding to infinity due to overshooting.

`lr = 0.1` is the sweet spot: MSE drops from 0.89 to 0.28 in 20 epochs, achieving **79% test accuracy** — matching logistic regression and SVM. This is also achieved in roughly **9 seconds**, compared to 220+ seconds for full batch GD. SGD is 20-40x faster because with batch size 10, it makes 10,000 weight updates across 20 epochs compared to only 500 for full batch GD, all at a fraction of the per-update cost.

## 4. Scalability Analysis

The models above were trained on a 5,000-example subset. Here we test which models can train on the **full dataset** (40,409 examples × 13,297 features) within 2 minutes.

Before running, we can reason about which models will fail:
- **Linear and Ridge Regression** require matrix operations on (d × d) matrices — O(d³) with d = 13,297. Ridge crashed Colab's RAM on the full dataset; the matrix XᵀX alone requires ~1.4GB.
- **Logistic Regression** (sklearn default solver) also crashed RAM on the full dataset for the same reasons.
- **Full Batch GD** took 222 seconds on 5,000 examples. Scaling to 40,409 examples (8x more data) would take approximately 30 minutes.

**LinearSVC and LinearRegressionSGD** are the candidates.

In [ ]:
import time

# LinearSVC on full training data
start = time.time()
svc_full = LinearSVC().fit(X_train, y_train)
print(f"SVC training time: {time.time() - start:.2f}s")
print("Train:", end='\t')
svc_metrics(X_train, y_train, svc_full)
print("Test: ", end='\t')
svc_metrics(X_test, y_test, svc_full)

In [ ]:
# SGD on full training data
start = time.time()
sgd_full = LinearRegressionSGD(lr=0.1, n_epochs=10, batch_size=32)
sgd_full.fit(X_train, y_train, verbose=True)
print(f"SGD training time: {time.time() - start:.2f}s")
print("Train:", end='\t')
linreg_metrics(X_train, y_train, sgd_full)
print("Test: ", end='\t')
linreg_metrics(X_test, y_test, sgd_full)

**Scalability Results:**

| Model | Scalable | Time | Test Accuracy |
|---|---|---|---|
| Linear Regression | ❌ | ~900s est. | — |
| Ridge Regression | ❌ | RAM crash | — |
| Logistic Regression | ❌ | RAM crash | — |
| Full Batch GD | ❌ | ~30min est. | — |
| **LinearSVC** | ✅ | **~7s** | **90%** |
| **SGD (custom)** | ✅ | **~18s** | **87%** |

**Best result: 90% test accuracy with LinearSVC on the full dataset** — an 11 percentage point improvement over the 5,000-sample subset, demonstrating the direct impact of training data volume. SVM is the clear winner: fastest model, best accuracy, and scales gracefully because it only updates weights based on support vectors near the decision boundary.

## 5. Key Findings & Takeaways

**Regularization matters.** Plain linear regression overfits severely on high-dimensional BoW features — 100% train accuracy but only 68% test. Ridge regression with appropriate α recovers 78% test accuracy by preventing large weights from memorizing noise.

**Loss function design is critical.** Logistic regression and SVM both outperform linear regression on test accuracy (79% vs 68%) because their loss functions are explicitly designed for binary classification. Despite completely different training philosophies — probabilistic vs geometric — they converge to the same test performance, consistent with both finding the same type of linear decision boundary.

**SGD scales, full batch GD does not.** Full batch GD took 220+ seconds on 5,000 examples. SGD achieved better accuracy (79% vs 78%) in 9 seconds — a 20-40x speedup — because mini-batch updates allow 10,000 weight updates in the time full batch GD makes 500.

**More data = better models.** SVM accuracy jumped from 79% on 5,000 examples to 90% on 40,000. The model architecture didn't change — only the amount of training signal available. This is one of the most consistent findings in ML.

**Scalability is a hard constraint, not just a preference.** Ridge and Logistic regression did not just run slowly on the full dataset — they crashed entirely due to memory limits. Choosing a scalable algorithm is a prerequisite before accuracy even becomes relevant.